In [1]:
from torch import nn
import numpy as np
import matplotlib as mlp
import pandas as pd
import h5py
from time import sleep
from torch.utils.data import Dataset, DataLoader, random_split, Subset

In [2]:
from pole_nn import H5Data
path = "./data.hdf5"
data = H5Data(path)

In [3]:
from sklearn.model_selection import train_test_split

labels = [int(float(group)) for group, _ in data.index_map]
seed = 1
train_idx, temp_idx = train_test_split(
    range(len(data.index_map)),
    train_size=0.8,
    stratify=labels,
    random_state=seed
)

temp_labels = [labels[i] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=seed,
)

train_sub = Subset(data, train_idx)
val_sub = Subset(data, val_idx)
test_sub = Subset(data, test_idx)

BATCH_SIZE = 32
train_dataloader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_sub, batch_size=BATCH_SIZE)
print(f"Length of train dataloader: {len(train_dataloader)}")
print(f"Length of val dataloader: {len(val_dataloader)}")
print(f"Length of test dataloader: {len(test_dataloader)}")

Length of train dataloader: 81261
Length of val dataloader: 10158
Length of test dataloader: 10158


In [ ]:
import torch
import torch.nn as nn
import optuna
from model import pole_nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate(model, dataloader, loss_fn):
    model.eval()
    total_loss = 0.0

    with torch.inference_mode():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            total_loss += loss_fn(model(x), y).item()

    return total_loss / len(dataloader)

def objective(trial):
    hidden_size_one = trial.suggest_int("hidden_size_one", 64, 128, step=16)
    hidden_size_two = trial.suggest_int("hidden_size_two", 64, 128, step=16)
    learning_rate = trial.suggest_float("lr", 1e-4, 1e-1, log=True)

    model = pole_nn(
        n_bins=25,
        n_classes=9029,
        hidden_one=hidden_size_one,
        hidden_two=hidden_size_two
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(10):
        model.train()

        for x, y in train_dataloader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()

        val_loss = evaluate(model, val_dataloader, loss_fn)
        trial.report(val_loss, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return val_loss

In [7]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

study.optimize(objective, n_trials=50)

print("Best loss:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-09-07 10:48:23,715] A new study created in memory with name: no-name-fed2b1be-e9fc-4537-8a35-11e8eeabb448
[I 2026-09-07 11:33:37,879] Trial 0 finished with value: 0.02771128228508896 and parameters: {'hidden_size': 64, 'lr': 0.0002137229354724628}. Best is trial 0 with value: 0.02771128228508896.
[I 2026-09-07 12:18:49,680] Trial 1 finished with value: 0.019665539041438796 and parameters: {'hidden_size': 128, 'lr': 0.0016439475931580242}. Best is trial 1 with value: 0.019665539041438796.
[I 2026-09-07 13:05:27,298] Trial 2 finished with value: 0.027930352728069884 and parameters: {'hidden_size': 80, 'lr': 0.0006569661602788282}. Best is trial 1 with value: 0.019665539041438796.
[I 2026-09-07 13:51:51,637] Trial 3 finished with value: 0.02790544611929217 and parameters: {'hidden_size': 80, 'lr': 0.0027543511753922024}. Best is trial 1 with value: 0.019665539041438796.
[W 2026-09-07 13:52:29,315] Trial 4 failed with parameters: {'hidden_size': 96, 'lr': 0.03828324572257007} becau

KeyboardInterrupt: 

In [8]:
study.trials_dataframe().to_csv("optuna_trials3.csv", index=False)
print(study.trials_dataframe())
print("Best:", study.best_params, study.best_value)

   number     value             datetime_start          datetime_complete  \
0       0  0.027711 2026-09-07 10:48:23.716611 2026-09-07 11:33:37.879102   
1       1  0.019666 2026-09-07 11:33:37.879809 2026-09-07 12:18:49.680816   
2       2  0.027930 2026-09-07 12:18:49.681299 2026-09-07 13:05:27.298255   
3       3  0.027905 2026-09-07 13:05:27.298754 2026-09-07 13:51:51.637491   
4       4       NaN 2026-09-07 13:51:51.638015 2026-09-07 13:52:29.315717   

                duration  params_hidden_size  params_lr     state  
0 0 days 00:45:14.162491                  64   0.000214  COMPLETE  
1 0 days 00:45:11.801007                 128   0.001644  COMPLETE  
2 0 days 00:46:37.616956                  80   0.000657  COMPLETE  
3 0 days 00:46:24.338737                  80   0.002754  COMPLETE  
4 0 days 00:00:37.677702                  96   0.038283      FAIL  
Best: {'hidden_size': 128, 'lr': 0.0016439475931580242} 0.019665539041438796


In [ ]:
import torch
from model import pole_nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = pole_nn(25, 9029, 128, 128)
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.00164)

In [11]:
epochs = 4
for epoch in range(epochs):
    
    train_loss, train_accuracy = 0, 0
    model.train()
    for batch, (x, y) in enumerate(train_dataloader):
        x, y = x.to(device), y.to(device)

        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        train_pred = torch.argmax(y_pred, dim=1)
        train_accuracy += (train_pred == y).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader) * BATCH_SIZE

    test_loss, test_accuracy = 0, 0
    model.eval()
    with torch.inference_mode():
        for x,y in (test_dataloader):
            x, y = x.to(device), y.to(device)

            y_pred = model(x)
            test_loss += loss_fn(y_pred, y).item()
            test_pred = torch.argmax(y_pred, dim=1)
            test_accuracy += (test_pred == y).sum().item()

        test_loss /= len(test_dataloader)
        test_accuracy /= len(test_dataloader) * BATCH_SIZE
    print(f"Epoch: {epoch}")
    print(f"\nTrain loss: {train_loss:.5f} | Train Accuracy: {train_accuracy:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_accuracy:.2f}\n")

Epoch: 0

Train loss: 0.27259 | Train Accuracy: 0.96065 | Test loss: 0.02782, Test acc: 0.99



KeyboardInterrupt: 

In [12]:
torch.save(model.state_dict(), "model_weights.pth")